# Utility Consumption Data Preprocessing

This notebook preprocesses raw utility consumption data by:
1. **Data enrichment**: Adding weather features (cloud cover) from Open-Meteo API
2. **Data cleaning**: Resolving datetime format inconsistencies and parsing
3. **Quality validation**: Checking for duplicates, missing timestamps, and invalid values
4. **Anomaly detection**: Identifying outliers and sensor malfunctions (flatline checks)
5. **Feature engineering**: Aggregating power consumption across feeders

The output is a cleaned, validated dataset ready for feature engineering and model training.

In [30]:
import pandas as pd
import numpy as np

## Step 1: Load Raw Data

Load the utility consumption CSV file and inspect its structure for initial data exploration.

In [31]:
Raw_Path = "Utility_consumption.csv"
df = pd.read_csv(Raw_Path)
df.head()

,Datetime,Temperature,Humidity,WindSpeed,F1_132KV_PowerConsumption,F2_132KV_PowerConsumption,F3_132KV_PowerConsumption
0,01-01-2017 00:00,6.559,73.8,0.083,34055.69620,16128.87538,20240.96386
1,01-01-2017 00:10,6.414,74.5,0.083,29814.68354,19375.07599,20131.08434
2,01-01-2017 00:20,6.313,74.5,0.080,29128.10127,19006.68693,19668.43373
3,01-01-2017 00:30,6.121,75.0,0.083,28228.86076,18361.09422,18899.27711
4,01-01-2017 00:40,5.921,75.7,0.081,27335.69620,17872.34043,18442.40964


In [32]:
# Enrich the raw data with Dhanbad cloud cover before preprocessing.
# Open-Meteo returns hourly cloud_cover as a percentage (0-100).
import json
from urllib.parse import urlencode
from urllib.request import urlopen

DHANBAD_LATITUDE = 23.7957
DHANBAD_LONGITUDE = 86.4304
raw_timestamps = pd.to_datetime(
    df["Datetime"].str.replace("-", "/", regex=False),
    format="%m/%d/%Y %H:%M",
)

# Fetch one extra day to interpolate the final intervals after 23:00.
start_date = raw_timestamps.min().date().isoformat()
end_date = (raw_timestamps.max().normalize() + pd.Timedelta(days=1)).date().isoformat()
params = {
    "latitude": DHANBAD_LATITUDE,
    "longitude": DHANBAD_LONGITUDE,
    "start_date": start_date,
    "end_date": end_date,
    "hourly": "cloud_cover",
    "timezone": "Asia/Kolkata",
}
url = "https://archive-api.open-meteo.com/v1/archive?" + urlencode(params)

with urlopen(url, timeout=60) as response:
    weather_data = json.load(response)

if "hourly" not in weather_data:
    raise RuntimeError(f"Open-Meteo response did not contain hourly data: {weather_data}")

hourly_cloud = pd.Series(
    weather_data["hourly"]["cloud_cover"],
    index=pd.to_datetime(weather_data["hourly"]["time"]),
    name="CloudCover",
    dtype="float64",
)
target_times = pd.DatetimeIndex(raw_timestamps)
combined_times = hourly_cloud.index.union(target_times).sort_values()
cloud_10min = (
    hourly_cloud.reindex(combined_times)
    .interpolate(method="time")
    .reindex(target_times)
)
df["CloudCover"] = cloud_10min.to_numpy()

if df["CloudCover"].isna().any():
    raise ValueError(f"CloudCover contains {df['CloudCover'].isna().sum()} missing values")
if not df["CloudCover"].between(0, 100).all():
    raise ValueError("CloudCover values must be between 0 and 100 percent")

print("Open-Meteo location:", weather_data.get("latitude"), weather_data.get("longitude"))
print("CloudCover range:", df["CloudCover"].min(), "to", df["CloudCover"].max())
df[["Datetime", "CloudCover"]].head(12)

Open-Meteo location: 23.796133 86.38477
CloudCover range: 0.0 to 100.0


,Datetime,CloudCover
0,01-01-2017 00:00,18.000000
1,01-01-2017 00:10,18.833333
2,01-01-2017 00:20,19.666667
3,01-01-2017 00:30,20.500000
4,01-01-2017 00:40,21.333333
5,01-01-2017 00:50,22.166667
6,01-01-2017 01:00,23.000000
7,01-01-2017 01:10,23.000000
8,01-01-2017 01:20,23.000000
9,01-01-2017 01:30,23.000000


In [33]:
df.describe()

,Temperature,Humidity,WindSpeed,F1_132KV_PowerConsumption,F2_132KV_PowerConsumption,F3_132KV_PowerConsumption,CloudCover
count,52416.000000,52416.000000,52416.000000,52416.000000,52416.000000,52416.000000,52416.000000
mean,18.810024,68.259518,1.959489,32344.970564,21042.509082,17835.406218,44.032032
std,5.815476,15.551177,2.348862,7130.562564,5201.465892,6622.165099,42.195886
min,3.247000,11.340000,0.050000,13895.696200,8560.081466,5935.174070,0.000000
25%,14.410000,58.310000,0.078000,26310.668692,16980.766032,13129.326630,0.000000
50%,18.780000,69.860000,0.086000,32265.920340,20823.168405,16415.117470,31.000000
75%,22.890000,81.400000,4.915000,37309.018185,24713.717520,21624.100420,95.000000
max,40.010000,94.800000,6.483000,52204.395120,37408.860760,47598.326360,100.000000


## Step 2: Enrich Data with Weather Features

**Rationale**: Cloud cover is an important external factor affecting energy consumption patterns. Solar irradiance (inferred from cloud cover) influences lighting loads and, indirectly, cooling loads.

**Approach**:
- Fetch hourly cloud cover data from Open-Meteo Archive API for Dhanbad location
- Interpolate hourly data to match the 10-minute resolution of utility consumption records
- Validate that cloud cover values are within valid 0-100% range

In [34]:
def fmt(s):
    return "dash" if "-" in s.split(" ")[0] else "slash"

## Step 3: Detect and Handle Datetime Format Inconsistencies

**Problem**: The raw data contains mixed datetime formats (both `MM-DD-YYYY` and `MM/DD/YYYY` with inconsistent separators).

**Approach**:
1. Detect format transitions by identifying whether each row uses dashes or slashes
2. Use a secondary heuristic: if the second date component (month when MM-first) exceeds 12, this confirms month-first ordering
3. Standardize all dates to a single format before parsing

In [35]:
df["fmt"] = df["Datetime"].apply(fmt)
print("Format counts:\n", df["fmt"].value_counts())

df["fmt_change"] = df["fmt"] != df["fmt"].shift(1)
print("\nFormat transition points:")
df.loc[df["fmt_change"], ["Datetime", "fmt"]]

Format counts:
 fmt
slash    31680
dash     20736
Name: count, dtype: int64

Format transition points:


,Datetime,fmt
0,01-01-2017 00:00,dash
1728,1/13/2017 0:00,slash
4464,02-01-2017 00:00,dash
6192,2/13/2017 0:00,slash
8496,03-01-2017 00:00,dash
10224,3/13/2017 0:00,slash
12960,04-01-2017 00:00,dash
14688,4/13/2017 0:00,slash
17280,05-01-2017 00:00,dash
19008,5/13/2017 0:00,slash


In [36]:
df.head()

,Datetime,Temperature,Humidity,WindSpeed,F1_132KV_PowerConsumption,F2_132KV_PowerConsumption,F3_132KV_PowerConsumption,CloudCover,fmt,fmt_change
0,01-01-2017 00:00,6.559,73.8,0.083,34055.69620,16128.87538,20240.96386,18.000000,dash,True
1,01-01-2017 00:10,6.414,74.5,0.083,29814.68354,19375.07599,20131.08434,18.833333,dash,False
2,01-01-2017 00:20,6.313,74.5,0.080,29128.10127,19006.68693,19668.43373,19.666667,dash,False
3,01-01-2017 00:30,6.121,75.0,0.083,28228.86076,18361.09422,18899.27711,20.500000,dash,False
4,01-01-2017 00:40,5.921,75.7,0.081,27335.69620,17872.34043,18442.40964,21.333333,dash,False


In [37]:
# A second date component above 12 proves that the first component is the month.
# This distinguishes MM-DD-YYYY from DD-MM-YYYY.

dash_second_num = df.loc[df["fmt"] == "dash", "Datetime"].str.split("-").str[1].astype(int)
slash_second_num = df.loc[df["fmt"] == "slash", "Datetime"].str.split("/").str[1].astype(int)

print("Dash-format rows with day > 12:", (dash_second_num > 12).sum())
print("Slash-format rows with day > 12:", (slash_second_num > 12).sum())
if not ((dash_second_num > 12).any() or (slash_second_num > 12).any()):
    raise ValueError("Cannot reliably infer whether Datetime is month-first or day-first")
# Dates with a second component above 12 confirm consistent month-first formatting.

df = df.drop(columns=["fmt", "fmt_change"])

Dash-format rows with day > 12: 0
Slash-format rows with day > 12: 31680


In [38]:
df.head()

,Datetime,Temperature,Humidity,WindSpeed,F1_132KV_PowerConsumption,F2_132KV_PowerConsumption,F3_132KV_PowerConsumption,CloudCover
0,01-01-2017 00:00,6.559,73.8,0.083,34055.69620,16128.87538,20240.96386,18.000000
1,01-01-2017 00:10,6.414,74.5,0.083,29814.68354,19375.07599,20131.08434,18.833333
2,01-01-2017 00:20,6.313,74.5,0.080,29128.10127,19006.68693,19668.43373,19.666667
3,01-01-2017 00:30,6.121,75.0,0.083,28228.86076,18361.09422,18899.27711,20.500000
4,01-01-2017 00:40,5.921,75.7,0.081,27335.69620,17872.34043,18442.40964,21.333333


In [39]:
#Fix Datetime
df["Datetime"] = pd.to_datetime(
    df["Datetime"].str.replace("-", "/", regex=False),
    format="%m/%d/%Y %H:%M"
)
df = df.sort_values("Datetime").reset_index(drop=True)

print("Date range:", df["Datetime"].min(), "to", df["Datetime"].max())
print("Total rows:", len(df))
df.head()

Date range: 2017-01-01 00:00:00 to 2017-12-30 23:50:00
Total rows: 52416


,Datetime,Temperature,Humidity,WindSpeed,F1_132KV_PowerConsumption,F2_132KV_PowerConsumption,F3_132KV_PowerConsumption,CloudCover
0,2017-01-01 00:00:00,6.559,73.8,0.083,34055.69620,16128.87538,20240.96386,18.000000
1,2017-01-01 00:10:00,6.414,74.5,0.083,29814.68354,19375.07599,20131.08434,18.833333
2,2017-01-01 00:20:00,6.313,74.5,0.080,29128.10127,19006.68693,19668.43373,19.666667
3,2017-01-01 00:30:00,6.121,75.0,0.083,28228.86076,18361.09422,18899.27711,20.500000
4,2017-01-01 00:40:00,5.921,75.7,0.081,27335.69620,17872.34043,18442.40964,21.333333


## Step 4: Parse Datetime and Sort Records

**Steps**:
1. Standardize all separators to `/` for consistent parsing
2. Convert to pandas datetime using `%m/%d/%Y %H:%M` format
3. Sort by datetime and reset index for proper time-series ordering

In [40]:
#check duplicates and missing timestamps
dup_count = df["Datetime"].duplicated().sum()
print("Duplicate timestamps:", dup_count)

full_range = pd.date_range(df["Datetime"].min(), df["Datetime"].max(), freq="10min")
missing_ts = full_range.difference(df["Datetime"])

print("Expected timestamps:", len(full_range))
print("Actual timestamps:", df["Datetime"].nunique())
print("Missing timestamps:", len(missing_ts))
if len(missing_ts) > 0:
    print(missing_ts[:10])

Duplicate timestamps: 0
Expected timestamps: 52416
Actual timestamps: 52416
Missing timestamps: 0


## Step 5: Validate Time Series Integrity

**Why this matters**: For time-series forecasting, a complete, unbroken sequence is critical:
- **Duplicates**: Multiple records at same timestamp can skew aggregations
- **Missing timestamps**: Gaps break continuity and break lagged features, train/test splits

**Approach**: 
- Count duplicate timestamps
- Generate expected 10-minute intervals and compare against actual records

In [41]:
#Check value validity (negatives, zeros, out-of-range)
value_cols = [
    "Temperature", "Humidity", "WindSpeed",
    "CloudCover", "F1_132KV_PowerConsumption",
    "F2_132KV_PowerConsumption", "F3_132KV_PowerConsumption"
]

for c in value_cols:
    neg = (df[c] < 0).sum()
    zero = (df[c] == 0).sum()
    print(f"{c} | negatives: {neg} | zeros: {zero} | min: {df[c].min()} | max: {df[c].max()}")

print("\nHumidity > 100 (invalid):", (df["Humidity"] > 100).sum())
print("CloudCover outside 0-100 (invalid):", (~df["CloudCover"].between(0, 100)).sum())
print("Total missing values:", df.isna().sum().sum())

Temperature | negatives: 0 | zeros: 0 | min: 3.247 | max: 40.01
Humidity | negatives: 0 | zeros: 0 | min: 11.34 | max: 94.8
WindSpeed | negatives: 0 | zeros: 0 | min: 0.05 | max: 6.483
CloudCover | negatives: 0 | zeros: 13889 | min: 0.0 | max: 100.0
F1_132KV_PowerConsumption | negatives: 0 | zeros: 0 | min: 13895.6962 | max: 52204.39512
F2_132KV_PowerConsumption | negatives: 0 | zeros: 0 | min: 8560.081466 | max: 37408.86076
F3_132KV_PowerConsumption | negatives: 0 | zeros: 0 | min: 5935.17407 | max: 47598.32636

Humidity > 100 (invalid): 0
CloudCover outside 0-100 (invalid): 0
Total missing values: 0


## Step 6: Validate Physical Value Constraints

**Why this matters**: Physically impossible values (negative consumption, humidity > 100%) indicate sensor errors, data entry mistakes, or transmission issues. Such values corrupt model training.

**Validations**:
- **Power consumption**: Must be non-negative (≥ 0)
- **Humidity**: Must be in range 0-100%
- **Cloud cover**: Must be in range 0-100% (from API)
- **Missing values**: Check for NaN/NULL entries

In [42]:
#Outlier check (rolling z-score)
power_cols = ["F1_132KV_PowerConsumption", "F2_132KV_PowerConsumption", "F3_132KV_PowerConsumption"]

for c in power_cols:
    roll_mean = df[c].rolling(144, center=True, min_periods=1).mean()  # 144 = 24h at 10-min freq
    roll_std = df[c].rolling(144, center=True, min_periods=1).std()
    z = (df[c] - roll_mean) / roll_std
    print(f"{c} rolling z-score outliers (|z|>4):", (z.abs() > 4).sum())

F1_132KV_PowerConsumption rolling z-score outliers (|z|>4): 0
F2_132KV_PowerConsumption rolling z-score outliers (|z|>4): 0
F3_132KV_PowerConsumption rolling z-score outliers (|z|>4): 0


## Step 7: Detect Statistical Outliers

**Why rolling z-score?**: Power consumption varies by time of day and season. A static threshold would miss context-dependent anomalies. Rolling z-score (using 24-hour windows) adapts to local patterns.

**Approach**: 
- Compute 24-hour rolling mean and std for each power consumption feeder
- Calculate rolling z-score at each point
- Flag extreme deviations (|z| > 4σ) as potential sensor errors or genuine demand spikes

In [43]:
#Flatline / stuck-sensor check
for c in power_cols:
    is_same = df[c].diff() == 0
    streak_id = (~is_same).cumsum()
    streak_lens = is_same.groupby(streak_id).sum()
    max_identical_values = streak_lens.max() + 1
    print(f"{c} -> max consecutive identical values: {max_identical_values}")

F1_132KV_PowerConsumption -> max consecutive identical values: 3
F2_132KV_PowerConsumption -> max consecutive identical values: 3
F3_132KV_PowerConsumption -> max consecutive identical values: 3


## Step 8: Detect Sensor Malfunctions (Flatline Test)

**Why this matters**: A sensor that reports identical values over many consecutive intervals is likely stuck, faulty, or offline. This pattern is distinct from legitimate stable consumption.

**Approach**: 
- Group consecutive records with identical values (no change in reading)
- Report the maximum streak length for each feeder
- This helps identify when maintenance or data imputation is needed

In [44]:
df["Total_PowerConsumption"] = df[power_cols].sum(axis=1)

OUT_PATH = "Utility_consumption_cleaned.csv"
df.to_csv(OUT_PATH, index=False)

print(f"Cleaned file saved to: {OUT_PATH}")
df.head()

Cleaned file saved to: Utility_consumption_cleaned.csv


,Datetime,Temperature,Humidity,WindSpeed,F1_132KV_PowerConsumption,F2_132KV_PowerConsumption,F3_132KV_PowerConsumption,CloudCover,Total_PowerConsumption
0,2017-01-01 00:00:00,6.559,73.8,0.083,34055.69620,16128.87538,20240.96386,18.000000,70425.53544
1,2017-01-01 00:10:00,6.414,74.5,0.083,29814.68354,19375.07599,20131.08434,18.833333,69320.84387
2,2017-01-01 00:20:00,6.313,74.5,0.080,29128.10127,19006.68693,19668.43373,19.666667,67803.22193
3,2017-01-01 00:30:00,6.121,75.0,0.083,28228.86076,18361.09422,18899.27711,20.500000,65489.23209
4,2017-01-01 00:40:00,5.921,75.7,0.081,27335.69620,17872.34043,18442.40964,21.333333,63650.44627


## Step 9: Aggregate and Export Cleaned Data

**Final processing**:
1. **Feature engineering**: Sum power consumption across all three 132kV feeders to create total consumption metric
2. **Export**: Save the cleaned, validated dataset for downstream feature engineering and model training

**Output format**: CSV file with validated datetime, enriched weather features, individual feeder consumption, and aggregated total consumption.